##### pulses.mat on HDAWG + SGS100A -- Wideband IQ Upconversion to 9.3 GHz\n\nPlays the shaped pulses stored in `pulses.mat` (same folder) through **HDAWG channels\n3 and 4** as a software-IQ pair, upconverted by the **SGS100A** in wideband IQ mode.\n\n- **HDAWG**: 200 MHz IF carrier, software IQ modulation, channels 3/4 (`SIGOUTS/2`,\n  `SIGOUTS/3`) as I/Q.\n- **SGS100A**: 9.3 GHz LO, `SOURce:IQ:WBSTate ON` (wideband IQ upconversion mode) --\n  RF output = 9.3 GHz +/- 200 MHz.\n- **Pulses played** (from `pulses.mat`, native 2.4 GSa/s playback -- 1 stored sample =\n  1 HDAWG sample, per the confirmed approach for this notebook):\n  - `my_pi_half_gauss_rf`, `my_pi_gauss_rf` -- the two shaped Gaussian pulses\n  - `my_seq_Hahn` -- the customer's pre-assembled Hahn-echo sequence, played as one\n    literal waveform (its free-evolution delay is fixed, baked into the array --\n    not swept on this hardware run)\n\nStructure follows `Rabi_IQ_Baseband_SGS_RTP.ipynb` (`ZI Blogs/Freq_Up_Conversion/Notebooks/`),\nwhich this notebook uses as its reference for device setup, calibration, and the\nSGS100A discovery/connect/IQ-enable pattern.\n\n**IQ impairment correction values (leakage, quadrature, gain) are frequency-specific**\nand not carried over from the reference notebook's 12 GHz calibration -- placeholders\nare left at nominal/zero below. Re-calibrate at 9.3 GHz using the same procedure as\n`CW_LO_Leakage_SGS_HDAWG.ipynb` before relying on these for a real measurement.

In [61]:
from laboneq.simple import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from RsInstrument import RsInstrument, BinFloatFormat
import numpy as np
import scipy.io as sio
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import base64
import io
import tempfile
import webbrowser


def show_in_browser(fig=None):
    """Render a matplotlib figure as a standalone HTML page and open it in the
    browser -- same helper used in Rabi_IQ_Baseband_SGS_RTP.ipynb."""
    fig = fig or plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode()
    html = (
        '<html><body style="margin:0">'
        f'<img src="data:image/png;base64,{img_b64}" style="max-width:100%">'
        "</body></html>"
    )
    with tempfile.NamedTemporaryFile(mode="w", suffix=".html", delete=False) as f:
        f.write(html)
        path = f.name
    webbrowser.open(f"file://{path}")
    plt.close(fig)


plt.show = show_in_browser

## 1. Device Setup

| Port | Role |
|------|------|
| SIGOUTS/2 | I (HDAWG channel 3) |
| SIGOUTS/3 | Q (HDAWG channel 4) |

`SIGOUTS/N` is 0-indexed, so `SIGOUTS/2`/`SIGOUTS/3` are hardware-panel channels 3/4 --
matching the reference notebook's own port pair, which already targets these same two
channels.

In [62]:
server_host = "localhost"
my_device_setup = DeviceSetup("hdawg_pulses_mat")
my_device_setup.add_dataserver(host=server_host, port="8004")

my_device_setup.add_instruments(
    HDAWG(uid="device_hdawg", address="DEV8538", device_options="HDAWG4/CNT/ME/MF/PC")
)

my_device_setup.add_connections(
    "device_hdawg",
    create_connection(to_signal="q0/drive_line", ports=["SIGOUTS/2", "SIGOUTS/3"], type="iq"),
)

## 2. Calibration -- 200 MHz Software IQ (HDAWG) + 9.3 GHz LO (SGS100A)

The HDAWG's software oscillator modulates the (real-valued) pulse envelopes onto a
200 MHz IF carrier, split across the I/Q channel pair. The SGS100A's LO (Section 9)
then upconverts that IF onto 9.3 GHz.

In [63]:
CARRIER_FREQ = 200e6   # 200 MHz IF carrier (HDAWG software IQ modulation)
LO_FREQ = 9.3e9        # SGS100A LO carrier (wideband IQ upconversion, Section 9)

calib = Calibration()
calib["q0/drive_line"] = SignalCalibration(
    oscillator=Oscillator(
        uid="drive_osc",
        frequency=CARRIER_FREQ,
        modulation_type=ModulationType.SOFTWARE,
    ),
    range=1.0,
)
my_device_setup.set_calibration(calib)

## 3. Load pulses.mat Pulses

Each waveform is squeezed to 1-D and rescaled to a max magnitude of 1.0 -- LabOneQ
pulse samples must be within [-1, 1], while `pulses.mat` stores them on the customer's
own arbitrary amplitude scale (up to ~157). Physical output amplitude is controlled
separately via the HDAWG output range / `exp.play(amplitude=...)`, not by these
rescaled sample values.

Playback is **native 2.4 GSa/s**: each stored array sample becomes exactly one HDAWG
sample, so pulse duration falls directly out of array length (confirmed approach for
this notebook, printed below).

In [64]:
PULSES_MAT_PATH = "pulses.mat"   # same folder as this notebook
pulses_data = sio.loadmat(PULSES_MAT_PATH)

HDAWG_SAMPLE_RATE_HZ = 2.4e9   # native HDAWG AWG core rate -- 1 stored sample = 1 AWG sample


def normalized(name):
    arr = pulses_data[name].squeeze().astype(float)
    return arr / np.max(np.abs(arr))


pi_half_samples = normalized("my_pi_half_gauss_rf")
pi_samples = normalized("my_pi_gauss_rf")
hahn_seq_samples = normalized("my_seq_Hahn")

pi_half_pulse = pulse_library.sampled_pulse_real(uid="pi_half_pulse_fau", samples=pi_half_samples)
pi_pulse = pulse_library.sampled_pulse_real(uid="pi_pulse_fau", samples=pi_samples)
hahn_seq_pulse = pulse_library.sampled_pulse_real(uid="hahn_seq_pulse_fau", samples=hahn_seq_samples)

for name, samples in [("pi/2 pulse", pi_half_samples), ("pi pulse", pi_samples), ("Hahn sequence", hahn_seq_samples)]:
    duration_ns = len(samples) / HDAWG_SAMPLE_RATE_HZ * 1e9
    print(f"{name:<14} {len(samples):>6} samples -> {duration_ns:8.1f} ns")

pi/2 pulse        251 samples ->    104.6 ns
pi pulse          251 samples ->    104.6 ns
Hahn sequence    3003 samples ->   1251.2 ns


/var/folders/dy/n7hsxx852xq2bytn1n_nl9vm0000gp/T/ipykernel_41814/2691512268.py:16: FutureWarning: The `sampled_pulse_real` function, along with `PulseSampledReal`, is deprecated. Please use `sampled_pulse` instead, as `sampled_pulse_real` now calls `sampled_pulse` internally.
  pi_half_pulse = pulse_library.sampled_pulse_real(uid="pi_half_pulse_fau", samples=pi_half_samples)
/var/folders/dy/n7hsxx852xq2bytn1n_nl9vm0000gp/T/ipykernel_41814/2691512268.py:17: FutureWarning: The `sampled_pulse_real` function, along with `PulseSampledReal`, is deprecated. Please use `sampled_pulse` instead, as `sampled_pulse_real` now calls `sampled_pulse` internally.
  pi_pulse = pulse_library.sampled_pulse_real(uid="pi_pulse_fau", samples=pi_samples)
/var/folders/dy/n7hsxx852xq2bytn1n_nl9vm0000gp/T/ipykernel_41814/2691512268.py:18: FutureWarning: The `sampled_pulse_real` function, along with `PulseSampledReal`, is deprecated. Please use `sampled_pulse` instead, as `sampled_pulse_real` now calls `sampled_p

## 4. Session

In [65]:
session = Session(device_setup=my_device_setup)
session.connect(do_emulation=False, ignore_version_mismatch=True)

[2026.09.28 16:49:47.799] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/avishekc/Library/CloudStorage/OneDrive-ZurichInstrumentsAG/Documents/ZI Supports/FAU_Berlin_TD_EPR/laboneq_output/log


[2026.09.28 16:49:47.801] INFO    Disconnecting from all devices and servers...
[2026.09.28 16:49:47.804] INFO    Successfully disconnected from all devices and servers.
[2026.09.28 16:49:47.830] INFO    VERSION: laboneq 26.7.0
[2026.09.28 16:49:47.831] INFO    Connecting to data server at localhost:8004
[2026.09.28 16:49:47.843] INFO    Connected to Zurich Instruments LabOne Data Server version 26.07.2.5 at localhost:8004
[2026.09.28 16:49:47.854] INFO    Configuring the device setup
[2026.09.28 16:49:47.862] INFO    The device setup is configured


## 5. SGS100A -- Auto-Discover on the Network

Same LXI/mDNS discovery helper as `Rabi_IQ_Baseband_SGS_RTP.ipynb`, falling back to
the last-known IP if discovery finds nothing.

In [66]:
import subprocess
import re

SGS_IP_FALLBACK = "10.42.192.35"   # Avishek's lab SGS100A -- used only if discovery fails


def discover_lxi_ip(name_contains, service="_lxi._tcp", timeout=4):
    """Find an LXI instrument on the local network via mDNS/Bonjour (`dns-sd`),
    matching `name_contains` against its advertised instance name.
    Returns (instance_name, ip) or (None, None) if not found."""

    def _run(args):
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
            return r.stdout
        except subprocess.TimeoutExpired as e:
            out = e.stdout or ""
            return out.decode(errors="replace") if isinstance(out, bytes) else out

    out = _run(["dns-sd", "-B", service, "local"])
    instance_name = None
    for line in out.splitlines():
        if not re.match(r"^\d{2}:\d{2}:\d{2}\.\d+\s+Add", line):
            continue
        parts = re.split(r"\s+", line.strip(), maxsplit=6)
        if len(parts) == 7 and name_contains.lower() in parts[6].lower():
            instance_name = parts[6]
            break
    if instance_name is None:
        return None, None

    out = _run(["dns-sd", "-L", instance_name, service, "local"])
    m = re.search(r"can be reached at ([\w.\-]+)\.local\.:", out)
    if not m:
        return instance_name, None
    hostname = m.group(1) + ".local"

    out = _run(["dns-sd", "-G", "v4", hostname])
    ip = None
    for line in out.splitlines():
        if "Add" in line:
            m2 = re.search(r"(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})", line)
            if m2:
                ip = m2.group(1)
                break
    return instance_name, ip


instance_name, discovered_ip = discover_lxi_ip("SGS100A")
if discovered_ip:
    SGS_IP = discovered_ip
    print(f"Found '{instance_name}' at {SGS_IP}")
else:
    SGS_IP = SGS_IP_FALLBACK
    print(f"Discovery found no SGS100A on the network -- falling back to last-known IP "
          f"{SGS_IP}. Update SGS_IP_FALLBACK by hand if this is wrong.")

Found 'Rohde & Schwarz RF Source SGS100A 110853' at 10.42.192.23


## 6. Connect to the SGS100A

In [67]:
SGS_IP = "10.42.192.23"
instr = RsInstrument(f"TCPIP::{SGS_IP}::hislip0", id_query=True, reset=True)
idn = instr.query_str("*IDN?")
print(idn + " is connected.")

Rohde&Schwarz,SGS100A,1416.0505k02/110853,4.2.366.0-5.00.232.70 is connected.


## 7. SGS100A -- Enable Wideband IQ Upconversion at 9.3 GHz

`SOURce:IQ:WBSTate ON` puts the SGS100A in wideband IQ mode, needed for the external
IQ bandwidth this 200 MHz IF requires. IQ impairment values below (`i_offset`,
`q_offset`, `quad`, `gain`) are nominal placeholders -- see the caveat in the title
cell; re-calibrate at `LO_FREQ` before trusting them for a real measurement.

In [68]:
i_offset = 0.0   # in %      -- nominal placeholder, re-calibrate at LO_FREQ
q_offset = 0.0   # in %      -- nominal placeholder, re-calibrate at LO_FREQ
quad = 0.0       # in deg    -- nominal placeholder, re-calibrate at LO_FREQ
gain = 0.0       # in dB     -- nominal placeholder, re-calibrate at LO_FREQ

instr.write_str("OUTPut:STATe ON")                            # RF output on
instr.write_str("SOURce:FREQuency:MODE CW")                   # continuous wave
instr.write_str(f"SOURce:FREQuency:FIXed {LO_FREQ:.0f}")       # 9.3 GHz LO
instr.write_str("SOURce:POWer:LEVel:IMMediate:AMPLitude 10")   # 10 dBm
instr.write_str("SOURce:IQ:WBSTate ON")                       # wideband IQ modulator path
instr.write_str("SOURce:IQ:IMPairment:STATe ON")               # enable impairment corrections
instr.write_str("SOURce:IQ:STATe ON")                         # IQ modulation on
instr.write_str(f"SOURce:IQ:IMPairment:LEAKage:I {i_offset}")
instr.write_str(f"SOURce:IQ:IMPairment:LEAKage:Q {q_offset}")
instr.write_str(f"SOURce:IQ:IMPairment:QUADrature:ANGLe {quad}")
instr.write_str(f"SOURce:IQ:IMPairment:IQRatio:MAGNitude {gain}")

## 8. Scope Capture Helpers (RTO6)

Shared connection settings and the arm/fetch functions used by both pulse sets below.
`CAPTURE_LENGTH` is passed in separately by each pulse set (Sections 9-12 and 13-16),
since the pi/2+pi pulses and the Hahn sequence need very different acquisition windows.

CH1 watches the drive signal, CH3 is the trigger source (Marker 1). Armed **non-blocking, before** `session.run()` in each pulse set's own "Run, Fetch, Plot"
section -- arming after `run()` would deadlock, since the marker will have already
fired by then (see the RS_RTO skill's Anti-pattern warning). RTO6-specific: trigger
level suffix `LEVel<n>` is the *channel number* (`LEVel3` for CH3), not a
trigger-event index; CH3's own waveform data can't be queried since it's used as a
trigger source (not needed here -- only CH1 is fetched).

In [69]:
SCOPE_IP   = "10.42.192.26"   # RTO6
CH1_RANGE  = 0.4              # CH1 full-scale range (V)
TRIG_LEVEL = 0.1              # CH3 trigger threshold (V)
TIMEOUT_MS = 10e3             # max wait for trigger (ms)


def setup_and_arm_scope(scope_ip, capture_length, trig_level, ch1_range):
    """Configure the RTO6 and arm for one CH3 rising-edge trigger (Marker 1).
    Returns the open scope object -- keep alive until fetch_scope_data().
    NON-BLOCKING: returns immediately after arming so the HDAWG can run."""
    scope = RsInstrument(f"TCPIP::{scope_ip}::hislip0", id_query=True, reset=True)
    scope.write_str("*CLS")

    scope.write_str("CHANnel1:STATe ON")
    scope.write_str(f"CHANnel1:RANGe {ch1_range}")
    scope.write_str("CHANnel1:COUPling DC")
    scope.write_str("CHANnel3:STATe ON")    # must be ON even when used only as trigger source
    scope.write_str("CHANnel3:COUPling DC")

    scope.write_str(f"TIMebase:SCALe {capture_length / 10}")

    scope.write_str("TRIGger1:SOURce CHANnel3")
    scope.write_str(f"TRIGger1:LEVel3:VALue {trig_level}")
    scope.write_str("TRIGger1:EDGE:SLOPe POSitive")
    scope.write_str("TRIGger1:MODE NORMal")

    scope.write_str("SINGle")   # arm -- non-blocking, returns immediately

    err = scope.query_str("SYSTem:ERRor:ALL?")
    if not err.startswith("0"):
        print(f"WARNING -- scope reported an error after arming: {err}")

    return scope


def fetch_scope_data(scope, timeout_ms=10e3):
    """Wait for the armed acquisition to complete, download CH1, close scope.
    Call this AFTER session.run() has sent the trigger pulse."""
    try:
        scope.visa_timeout = timeout_ms
        scope.query_str("*OPC?")   # blocks until SINGle acquisition is done

        scope.visa_timeout = 30_000
        scope.write_str("FORM REAL,32")
        scope.bin_float_numbers_format = BinFloatFormat.Single_4bytes
        y1 = np.array(scope.query_bin_or_ascii_float_list("CHANnel1:DATA?"))
        sr = float(scope.query_str("ACQuire:SRATe?"))

        t = np.arange(len(y1)) / sr * 1e6   # us
        return t, y1, sr
    finally:
        scope.write_str("SYSTem:KLOCk OFF")
        scope.close()

## 9. Pi/2 and Pi Pulses -- Experiment

Plays the pi/2 pulse then the pi pulse back to back. Marker 1 fires at the start so
the RTO6 can trigger on it.

In [70]:
exp_pi_pulses = Experiment(
    uid="Pi_Half_Pi_Pulses",
    signals=[ExperimentSignal("drive")],
)

exp_pi_pulses.set_signal_map({
    "drive": my_device_setup.logical_signal_groups["q0"].logical_signals["drive_line"],
})

with exp_pi_pulses.acquire_loop_rt(
    uid="shots",
    count=1,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp_pi_pulses.section(uid="pi_half", alignment=SectionAlignment.LEFT):
        exp_pi_pulses.play(signal="drive", pulse=pi_half_pulse, marker={"marker1": {"enable": True}})

    with exp_pi_pulses.section(uid="pi", play_after="pi_half"):
        exp_pi_pulses.play(signal="drive", pulse=pi_pulse)

    with exp_pi_pulses.section(uid="relax", play_after="pi"):
        exp_pi_pulses.delay(signal="drive", time=2000e-9)

## 10. Pi/2 and Pi Pulses -- Compile & Simulate

In [71]:
compiled_exp_pi_pulses = session.compile(exp_pi_pulses)
plot_simulation(compiled_exp_pi_pulses, start_time=0, length=500e-9)

[2026.09.28 16:50:19.268] INFO    Starting LabOne Q Compiler run...
[2026.09.28 16:50:19.274] INFO    Schedule completed. [0.001 s]
[2026.09.28 16:50:19.283] INFO    Code generation completed. [0.009 s]
[2026.09.28 16:50:19.284] INFO    Completed compilation step 1 of 1. [0.012 s]
[2026.09.28 16:50:19.289] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:19.290] INFO      Device         AWG   SeqC LOC   CT entries   Waveforms   Samples  
[2026.09.28 16:50:19.291] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:19.291] INFO      device_hdawg     0          3            1           0         0  
[2026.09.28 16:50:19.291] INFO      device_hdawg     1          8            2           1      1024  
[2026.09.28 16:50:19.292] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:19.292] INFO      TOTAL                      11            3                  1024

## 11. Pi/2 and Pi Pulses -- Arm RTO6 Scope

~500 ns window -- comfortably frames both ~105 ns pulses back to back.

In [72]:
PI_PULSES_CAPTURE_LENGTH = 500e-9   # acquisition window (s)

scope_handle_pi = setup_and_arm_scope(SCOPE_IP, PI_PULSES_CAPTURE_LENGTH, TRIG_LEVEL, CH1_RANGE)

## 12. Pi/2 and Pi Pulses -- Run, Fetch, Plot

In [73]:
results_pi_pulses = session.run(compiled_exp_pi_pulses)   # HDAWG fires Marker 1 -> RTO6 captures

t_pi, y1_pi, sr_pi = fetch_scope_data(scope_handle_pi, TIMEOUT_MS)

plt.figure(figsize=(10, 3))
plt.plot(t_pi, y1_pi * 1e3, lw=0.8, color="steelblue")
plt.ylabel("CH1 (mV)")
plt.xlabel("Time (us)")
plt.title(f"RTO6 capture -- pi/2 + pi pulses, window {PI_PULSES_CAPTURE_LENGTH * 1e6:.2f} us, trigger CH3")
plt.tight_layout()
plt.show()

[2026.09.28 16:50:32.361] INFO    Starting near-time execution...
[2026.09.28 16:50:32.439] INFO    Finished near-time execution.


## 13. Hahn Echo Sequence -- Experiment

Plays the full pre-baked `my_seq_Hahn` waveform as one literal pulse. Marker 1 fires
at the start so the RTO6 can trigger on it.

In [74]:
exp_hahn = Experiment(
    uid="Hahn_Echo_Sequence",
    signals=[ExperimentSignal("drive")],
)

exp_hahn.set_signal_map({
    "drive": my_device_setup.logical_signal_groups["q0"].logical_signals["drive_line"],
})

with exp_hahn.acquire_loop_rt(
    uid="shots",
    count=1,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp_hahn.section(uid="hahn_seq", alignment=SectionAlignment.LEFT):
        exp_hahn.play(signal="drive", pulse=hahn_seq_pulse, marker={"marker1": {"enable": True}})

    with exp_hahn.section(uid="relax", play_after="hahn_seq"):
        exp_hahn.delay(signal="drive", time=2000e-9)

## 14. Hahn Echo Sequence -- Compile & Simulate

In [75]:
compiled_exp_hahn = session.compile(exp_hahn)
plot_simulation(compiled_exp_hahn, start_time=0, length=2e-6)

[2026.09.28 16:50:39.372] INFO    Starting LabOne Q Compiler run...
[2026.09.28 16:50:39.373] INFO    Schedule completed. [0.000 s]
[2026.09.28 16:50:39.376] INFO    Code generation completed. [0.002 s]
[2026.09.28 16:50:39.377] INFO    Completed compilation step 1 of 1. [0.003 s]
[2026.09.28 16:50:39.378] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:39.379] INFO      Device         AWG   SeqC LOC   CT entries   Waveforms   Samples  
[2026.09.28 16:50:39.379] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:39.380] INFO      device_hdawg     0          3            1           0         0  
[2026.09.28 16:50:39.380] INFO      device_hdawg     1          8            2           1      6016  
[2026.09.28 16:50:39.381] INFO     ────────────────────────────────────────────────────────────────── 
[2026.09.28 16:50:39.382] INFO      TOTAL                      11            3                  6016

## 15. Hahn Echo Sequence -- Arm RTO6 Scope

2 us window -- covers the ~1.25 us Hahn sequence with margin.

In [78]:
HAHN_CAPTURE_LENGTH = 5e-6   # acquisition window (s)

scope_handle_hahn = setup_and_arm_scope(SCOPE_IP, HAHN_CAPTURE_LENGTH, TRIG_LEVEL, CH1_RANGE)

## 16. Hahn Echo Sequence -- Run, Fetch, Plot

In [79]:
results_hahn = session.run(compiled_exp_hahn)   # HDAWG fires Marker 1 -> RTO6 captures

t_hahn, y1_hahn, sr_hahn = fetch_scope_data(scope_handle_hahn, TIMEOUT_MS)

plt.figure(figsize=(10, 3))
plt.plot(t_hahn, y1_hahn * 1e3, lw=0.8, color="darkorange")
plt.ylabel("CH1 (mV)")
plt.xlabel("Time (us)")
plt.title(f"RTO6 capture -- Hahn echo sequence, window {HAHN_CAPTURE_LENGTH * 1e6:.2f} us, trigger CH3")
plt.tight_layout()
plt.show()

[2026.09.28 16:51:22.902] INFO    Starting near-time execution...
[2026.09.28 16:51:22.968] INFO    Finished near-time execution.
